# PF4 Replication Analysis using FinnGen GWAS Summary Statistics

This notebook processes FinnGen GWAS summary statistics for heart failure, coronary artery disease, and myocardial infarction. FinnGen is used as an independent replication resource to check whether PF4-related variants also appear in comparable cardiovascular phenotypes.

In [1]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
region_path = Path("../data/interim/ensembl/region.json")

finngen_hf_path = Path("../data/raw/finngen/finngen_heart_failure.tsv")
finngen_cad_path = Path("../data/raw/finngen/finngen_cad.tsv")
finngen_mi_path = Path("../data/raw/finngen/finngen_mi.tsv")

out_dir = Path("../data/interim/finngen")
out_dir.mkdir(parents=True, exist_ok=True)

out_hf_csv = out_dir / "heart_failure_associations.csv"
out_cad_csv = out_dir / "cad_associations.csv"
out_mi_csv = out_dir / "mi_associations.csv"
out_summary_json = out_dir / "summary.json"

In [3]:
with open(region_path, "r") as f:
    region = json.load(f)

chromosome = str(region["chromosome"]).replace("chr", "")
region_start = int(region["region_start"])
region_end = int(region["region_end"])
assembly = region["assembly_name"]

print("Gene:", region["gene_symbol"])
print("Assembly:", assembly)
print("PF4 region:", f"chr{chromosome}:{region_start}-{region_end}")

Gene: PF4
Assembly: GRCh38
PF4 region: chr4:73930811-74032027


In [4]:
for path in [finngen_hf_path, finngen_cad_path, finngen_mi_path]:
    if not path.exists():
        raise FileNotFoundError(f"FinnGen file not found: {path}")

In [5]:
finngen_preview = pd.read_csv(
    finngen_hf_path,
    sep="\t",
    nrows=5
)

finngen_preview

,#chrom,pos,ref,alt,rsids,nearest_genes,pval,mlogp,beta,sebeta,af_alt,af_alt_cases,af_alt_controls
0,1,13668,G,A,rs2691328,DDX11L1,0.529105,0.276458,0.061669,0.097985,0.005986,0.006138,0.005974
1,1,14506,G,A,rs1240557819,WASH7P,0.936567,0.028461,-0.008487,0.106644,0.004639,0.004614,0.004641
2,1,14521,C,T,rs1378626194,WASH7P,0.123522,0.908256,0.326778,0.212172,0.001083,0.001113,0.001081
3,1,14717,G,A,rs377122907,WASH7P,0.933506,0.029883,-0.036576,0.438374,0.000237,0.000244,0.000236
4,1,14773,C,T,rs878915777,WASH7P,0.788099,0.103419,0.015343,0.057084,0.014208,0.014380,0.014194


In [6]:
def normalize_chromosome(value):
    value = str(value).strip()
    value = value.replace("chr", "")

    if value.endswith(".0"):
        value = value[:-2]

    return value

In [7]:
def process_finngen_dataset(input_path):
    usecols = [
        "#chrom",
        "pos",
        "ref",
        "alt",
        "rsids",
        "nearest_genes",
        "pval",
        "mlogp",
        "beta",
        "sebeta",
        "af_alt",
        "af_alt_cases",
        "af_alt_controls"
    ]

    matched_chunks = []

    for chunk in pd.read_csv(
        input_path,
        sep="\t",
        usecols=usecols,
        chunksize=500_000,
        low_memory=False
    ):
        chunk = chunk.rename(columns={"#chrom": "chrom"})

        chunk["chrom_filter"] = chunk["chrom"].apply(normalize_chromosome)
        chunk["pos_filter"] = pd.to_numeric(chunk["pos"], errors="coerce")

        matched = chunk[
            (chunk["chrom_filter"] == chromosome)
            & (chunk["pos_filter"].between(region_start, region_end))
        ].copy()

        matched = matched.drop(columns=["chrom_filter", "pos_filter"])

        if not matched.empty:
            matched_chunks.append(matched)

    if matched_chunks:
        finngen_associations_df = pd.concat(matched_chunks, ignore_index=True)
    else:
        finngen_associations_df = pd.DataFrame(columns=[
            col.replace("#chrom", "chrom") for col in usecols
        ])

    rows_before_deduplication = len(finngen_associations_df)

    finngen_associations_df = finngen_associations_df.drop_duplicates()

    rows_after_deduplication = len(finngen_associations_df)
    duplicate_rows_removed = rows_before_deduplication - rows_after_deduplication

    finngen_associations_df = finngen_associations_df.rename(columns={
        "chrom": "FINNGEN_CHR",
        "pos": "FINNGEN_BP",
        "ref": "FINNGEN_ref",
        "alt": "FINNGEN_alt",
        "rsids": "FINNGEN_rsids",
        "nearest_genes": "FINNGEN_nearest_genes",
        "pval": "FINNGEN_p_value",
        "mlogp": "FINNGEN_mlogp",
        "beta": "FINNGEN_beta",
        "sebeta": "FINNGEN_std_error",
        "af_alt": "FINNGEN_alt_allele_freq",
        "af_alt_cases": "FINNGEN_alt_allele_freq_cases",
        "af_alt_controls": "FINNGEN_alt_allele_freq_controls"
    })

    numeric_cols = [
        "FINNGEN_BP",
        "FINNGEN_p_value",
        "FINNGEN_mlogp",
        "FINNGEN_beta",
        "FINNGEN_std_error",
        "FINNGEN_alt_allele_freq",
        "FINNGEN_alt_allele_freq_cases",
        "FINNGEN_alt_allele_freq_controls"
    ]

    for col in numeric_cols:
        finngen_associations_df[col] = pd.to_numeric(
            finngen_associations_df[col],
            errors="coerce"
        )

    finngen_associations_df = finngen_associations_df.sort_values(
        "FINNGEN_p_value",
        ascending=True,
        na_position="last"
    )

    return finngen_associations_df, duplicate_rows_removed

In [8]:
finngen_hf_df, hf_duplicate_rows_removed = process_finngen_dataset(
    input_path=finngen_hf_path
)

finngen_hf_df.shape

(647, 13)

In [9]:
finngen_cad_df, cad_duplicate_rows_removed = process_finngen_dataset(
    input_path=finngen_cad_path
)

finngen_cad_df.shape

(647, 13)

In [10]:
finngen_mi_df, mi_duplicate_rows_removed = process_finngen_dataset(
    input_path=finngen_mi_path
)

finngen_mi_df.shape

(647, 13)

In [11]:
finngen_hf_df.to_csv(out_hf_csv, index=False)
finngen_cad_df.to_csv(out_cad_csv, index=False)
finngen_mi_df.to_csv(out_mi_csv, index=False)

print("Saved:", out_hf_csv)
print("Saved:", out_cad_csv)
print("Saved:", out_mi_csv)

Saved: ../data/interim/finngen/heart_failure_associations.csv
Saved: ../data/interim/finngen/cad_associations.csv
Saved: ../data/interim/finngen/mi_associations.csv


In [12]:
summary = {
    "source_dataset": "FinnGen R12 summary statistics",
    "region": f"chr{chromosome}:{region_start}-{region_end}",
    "region_assembly": assembly,
    "heart_failure": {
        "phenotype": "Heart failure",
        "finngen_endpoint": "I9_HEARTFAIL",
        "associations": int(len(finngen_hf_df)),
        "unique_rsIDs": int(finngen_hf_df["FINNGEN_rsids"].nunique()),
    },
    "cad": {
        "phenotype": "Coronary artery disease",
        "finngen_endpoint": "I9_CHD",
        "associations": int(len(finngen_cad_df)),
        "unique_rsIDs": int(finngen_cad_df["FINNGEN_rsids"].nunique()),
    },
    "mi": {
        "phenotype": "Myocardial infarction",
        "finngen_endpoint": "I9_MI_STRICT",
        "associations": int(len(finngen_mi_df)),
        "unique_rsIDs": int(finngen_mi_df["FINNGEN_rsids"].nunique()),
    }
}

with open(out_summary_json, "w") as f:
    json.dump(summary, f, indent=4)

summary

{'source_dataset': 'FinnGen R12 summary statistics',
 'region': 'chr4:73930811-74032027',
 'region_assembly': 'GRCh38',
 'heart_failure': {'phenotype': 'Heart failure',
  'finngen_endpoint': 'I9_HEARTFAIL',
  'associations': 647,
  'unique_rsIDs': 606},
 'cad': {'phenotype': 'Coronary artery disease',
  'finngen_endpoint': 'I9_CHD',
  'associations': 647,
  'unique_rsIDs': 606},
 'mi': {'phenotype': 'Myocardial infarction',
  'finngen_endpoint': 'I9_MI_STRICT',
  'associations': 647,
  'unique_rsIDs': 606}}